# Mirror AraMS-28k-HTR (Zenodo 2.4 GB) → Hugging Face

One-shot data-prep. Downloads the AraMS-28k-HTR release from Zenodo
(`10.5281/zenodo.21499649`, ~2.4 GB — slow from a laptop, fast from Colab),
parses its `images/{book}_{page}_{line}.png` + `.gt.txt` pairs and the
train/val/test manifests, and pushes a streaming-ready dataset to
**`<your-hf-user>/arams28k-htr-lines`** so every other notebook can
`load_dataset` it like the KITAB / RASAM corpora — no 2.4 GB local dependency.

It also writes a **Kraken-format tar** (`images/` tree + manifests) to Google
Drive for `08_colab_train_ctc_baseline.ipynb`.

**Before running:** add the Colab secret `HF_TOKEN` (write scope). No GPU
needed — `Runtime → Change runtime type → CPU` is fine. ~10–20 min.

## 1. Auth

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
HF_USER = whoami()['name']
REPO = f'{HF_USER}/arams28k-htr-lines'
print('will push to', REPO)

## 2. Mount Drive (for the Kraken-format copy)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
WORK = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench')
(WORK / 'data').mkdir(parents=True, exist_ok=True)

## 3. Download the archive (resumable)

`wget -c` resumes a partial file, so re-running this cell after a disconnect
picks up where it left off.

In [ ]:
ZURL = 'https://zenodo.org/api/records/21499649/files/AraMS-28k-HTR.zip/content'
!mkdir -p /content/arams_dl
!wget -c -q --show-progress -O /content/arams_dl/AraMS-28k-HTR.zip '{ZURL}'
import os
sz = os.path.getsize('/content/arams_dl/AraMS-28k-HTR.zip')
print(f'{sz/1e9:.2f} GB')
assert sz > 2_300_000_000, 'download looks short — re-run this cell to resume'

## 4. Extract + inspect the real layout

In [ ]:
import zipfile, pathlib, json, collections
ROOT = pathlib.Path('/content/arams')
if not ROOT.exists():
    with zipfile.ZipFile('/content/arams_dl/AraMS-28k-HTR.zip') as z:
        z.extractall('/content/arams')
# the zip may or may not contain a top AraMS-28k-HTR/ dir — find the one with images/
BASE = next(p.parent for p in ROOT.rglob('images') if p.is_dir())
print('dataset base:', BASE)
print(sorted(p.name for p in BASE.iterdir()))
for f in ('dataset_stats.json', 'split_used.json'):
    fp = BASE / f
    if fp.exists(): print('\n==', f, '==\n', fp.read_text()[:1200])
print('\n== metadata.csv (head) ==')
print((BASE / 'metadata.csv').read_text().splitlines()[:5])
n_png = len(list((BASE / 'images').glob('*.png')))
print('\nline images:', n_png)

## 5. Parse → per-split records

Filename is `{book_id}_{page_id}_{line_idx}.png`; split membership comes from
the three `*_manifest.txt` files; `metadata.csv` is joined for `script` /
`source` / `zone` (whatever columns it actually has — printed above).

In [ ]:
import csv, pathlib
IMAGES = BASE / 'images'

def _manifest_ids(name):
    fp = BASE / name
    if not fp.exists():
        return set()
    out = set()
    for ln in fp.read_text().splitlines():
        ln = ln.strip()
        if not ln:
            continue
        # a manifest line may be a bare id, a filename, or a path
        stem = pathlib.Path(ln).name
        for suf in ('.png', '.gt.txt'):
            if stem.endswith(suf):
                stem = stem[:-len(suf)]
        out.add(stem)
    return out

SPLIT_IDS = {s: _manifest_ids(f'{s}_manifest.txt') for s in ('train', 'val', 'test')}
print({s: len(v) for s, v in SPLIT_IDS.items()})

# metadata.csv -> dict keyed by sample id (first col that looks like an id)
META = {}
with open(BASE / 'metadata.csv', newline='') as fh:
    r = csv.DictReader(fh)
    cols = r.fieldnames
    idcol = next((c for c in cols if c.lower() in ('id', 'sample_id', 'filename', 'image', 'name')), cols[0])
    for row in r:
        key = pathlib.Path(str(row[idcol])).name
        for suf in ('.png', '.gt.txt'):
            if key.endswith(suf): key = key[:-len(suf)]
        META[key] = row
print('metadata columns:', cols)

def _split_of(sid):
    for s, ids in SPLIT_IDS.items():
        if sid in ids:
            return s
    return None

def _parse_name(stem):
    # {book_id}_{page_id}_{line_idx} — line_idx is the last _-field, page_id the
    # one before, book_id everything before that (book ids can contain _)
    parts = stem.split('_')
    if len(parts) >= 3:
        return '_'.join(parts[:-2]), parts[-2], parts[-1]
    return stem, '', '0'

records = {'train': [], 'val': [], 'test': []}
missing_split = 0
for png in sorted(IMAGES.glob('*.png')):
    stem = png.stem
    gt = png.with_suffix('.gt.txt')
    if not gt.exists():
        continue
    text = gt.read_text(encoding='utf-8').strip()
    if not text:
        continue
    sp = _split_of(stem)
    if sp is None:
        missing_split += 1
        continue
    book, page, line = _parse_name(stem)
    m = META.get(stem, {})
    rec = {
        'sample_id': stem, 'image_path': str(png), 'text': text,
        'book_id': book, 'page_id': page, 'line_idx': line,
        'script': (m.get('script') or m.get('style') or m.get('hand') or '').lower(),
        'zone': (m.get('zone') or m.get('region') or m.get('line_type') or 'main').lower(),
        'source': m.get('source') or m.get('book') or book,
    }
    # carry any confidence / validation-level column verbatim if present
    for c in m:
        cl = c.lower()
        if 'conf' in cl and 'confidence' not in rec:
            rec['confidence'] = m[c]
        if cl in ('validation_level', 'validation', 'pv_lv', 'level'):
            rec['validation_level'] = m[c]
    records[sp].append(rec)
print({s: len(v) for s, v in records.items()}, 'no-split:', missing_split)
print('sample record:', records['test'][0])

## 6. Build the DatasetDict and push

Columns: `image` (decoded PNG), `text` (the released — already normalised —
transcription), `book_id`, `page_id`, `line_idx`, `script`, `zone`, `source`,
and `confidence` / `validation_level` **iff** `metadata.csv` carried them.

In [ ]:
from datasets import Dataset, DatasetDict, Image as HFImage, Features, Value

def _to_ds(rows):
    keys = sorted({k for r in rows for k in r})
    keys = [k for k in keys if k != 'sample_id']
    d = {k: [r.get(k) for r in rows] for k in keys}
    d['image'] = [r['image_path'] for r in rows]
    del d['image_path']
    ds = Dataset.from_dict(d).cast_column('image', HFImage())
    return ds

dd = DatasetDict({s: _to_ds(records[s]) for s in ('train', 'val', 'test') if records[s]})
print(dd)
dd.push_to_hub(REPO, private=False)
print('pushed ->', REPO)

## 7. Verify the mirror streams

In [ ]:
from datasets import load_dataset
chk = load_dataset(REPO, split='test', streaming=True)
row = next(iter(chk))
print({k: (v if k != 'image' else v.size) for k, v in row.items()})

## 8. Kraken-format copy to Drive

`08_colab_train_ctc_baseline.ipynb` reads this instead of re-downloading:
the `images/` tree (png + `.gt.txt`) plus the manifests, tarred to Drive.

In [ ]:
import tarfile, pathlib
OUT = WORK / 'data' / 'arams28k-htr-kraken.tar'
if not OUT.exists():
    with tarfile.open(OUT, 'w') as tf:
        tf.add(BASE / 'images', arcname='images')
        for m in ('train_manifest.txt', 'val_manifest.txt', 'test_manifest.txt',
                  'metadata.csv', 'split_used.json', 'dataset_stats.json'):
            if (BASE / m).exists():
                tf.add(BASE / m, arcname=m)
    print('wrote', OUT, f'({OUT.stat().st_size/1e9:.2f} GB)')
else:
    print('already present:', OUT)

## Notes

- The released `.gt.txt` is the corpus's **normalised** transcription. If a
  later experiment needs the *raw diacritised* form or per-line
  page-validated / line-validated flags, those live in the **full AraMS-28k**
  release (`10.5281/zenodo.22095333`), not in `-HTR` — mirror that separately
  when Tier-0 experiments 0.4 / 0.5 need it.
- Re-run any cell after a disconnect; the download resumes and the push
  overwrites the same repo.